# EXP 8: LEFT FACTORING AND RDP

This Google Colab notebook contains two independently executable C programs:

1. Left factoring of a context-free grammar (CFG)
2. Recursive-descent parsing of arithmetic expressions

> Run each cell in order. Each C source file is created inside Colab, compiled with GCC, and tested with sample input.

## EXP 8 a. Left Factoring of a CFG

### Aim

To write a C program that reads productions of a context-free grammar, identifies alternatives having a common prefix, applies immediate left factoring when required, and displays the resulting grammar.

### Concept

A grammar requires **left factoring** when two or more alternatives of the same nonterminal begin with a common prefix. A predictive parser cannot select the correct alternative using the next input symbol while that prefix remains common.



In [ ]:
%%writefile left_factoring.czz
#include <stdio.h>
#include <string.h>
#include <stdlib.h>

#define MAX_PROD 20
#define MAX_ALT 20
#define MAX_LEN 100

int common_prefix_len(const char *a, const char *b) {
    int i = 0;
    while (a[i] && b[i] && a[i] == b[i]) i++;
    return i;
}

int main(void) {
    int n;
    char line[MAX_LEN], lhs[MAX_PROD][20], rhs[MAX_PROD][MAX_LEN];

    printf("Enter number of productions: ");
    if (scanf("%d", &n) != 1 || n < 1 || n > MAX_PROD) return 1;
    printf("Enter productions (example: S->iEtS|iEtSeS|a):\n");

    for (int p = 0; p < n; p++) {
        scanf("%99s", line);
        char *arrow = strstr(line, "->");
        if (!arrow) { printf("Invalid production.\n"); return 1; }
        *arrow = '\0';
        strcpy(lhs[p], line);
        strcpy(rhs[p], arrow + 2);
    }

    printf("\nResulting grammar:\n");
    for (int p = 0; p < n; p++) {
        char temp[MAX_LEN], alt[MAX_ALT][MAX_LEN];
        int count = 0;
        strcpy(temp, rhs[p]);
        char *tok = strtok(temp, "|");
        while (tok && count < MAX_ALT) {
            strcpy(alt[count++], tok);
            tok = strtok(NULL, "|");
        }

        int best = 0, seed = -1;
        for (int i = 0; i < count; i++)
            for (int j = i + 1; j < count; j++) {
                int k = common_prefix_len(alt[i], alt[j]);
                if (k > best) { best = k; seed = i; }
            }

        if (best == 0) {
            printf("%s->%s\n", lhs[p], rhs[p]);
            continue;
        }

        char prefix[MAX_LEN];
        strncpy(prefix, alt[seed], best);
        prefix[best] = '\0';
        int grouped[MAX_ALT] = {0}, group_count = 0;
        for (int i = 0; i < count; i++) {
            if (strncmp(alt[i], prefix, best) == 0) { grouped[i] = 1; group_count++; }
        }

        printf("%s->%s%s'", lhs[p], prefix, lhs[p]);
        for (int i = 0; i < count; i++)
            if (!grouped[i]) printf("|%s", alt[i]);
        printf("\n%s'->", lhs[p]);
        int first = 1;
        for (int i = 0; i < count; i++) {
            if (grouped[i]) {
                if (!first) printf("|");
                printf("%s", alt[i][best] ? alt[i] + best : "#");
                first = 0;
            }
        }
        printf("\n");
    }
    return 0;
}


Writing left_factoring.c


### Compile and run with sample input

The sample grammar contains a common prefix `iEtS` in the first two alternatives.

In [2]:
!gcc  left_factoring.c -o left_factoring
!printf '2\nS->iEtS|iEtSeS|a\nA->b|c\n' | ./left_factoring

Enter number of productions: Enter productions (example: S->iEtS|iEtSeS|a):

Resulting grammar:
S->iEtSS'|a
S'->#|eS
A->b|c


### Expected output

```text
Enter number of productions: Enter productions (example: S->iEtS|iEtSeS|a):

Resulting grammar:
S->iEtSS'|a
S'->#|eS
A->b|c
```

Here `#` represents epsilon. The program performs one immediate left-factoring step for the longest common prefix in each production.

## EXP 8b. Implement Recursive-Descent Parser

### Aim

To implement a recursive-descent parser in C for a given expression grammar and determine whether an input string belongs to the language generated by the grammar.

### Concept

Recursive-descent parsing is a **top-down parsing** technique. Each nonterminal is implemented as a C function. The functions consume input symbols according to the productions. The input is accepted only when the start-symbol function succeeds and the entire input has been consumed.

The grammar must be free from left recursion and should normally be left factored.

### Grammar used

$$E \\rightarrow TE'$$
$$E' \\rightarrow +TE' \\mid \\varepsilon$$
$$T \\rightarrow FT'$$
$$T' \\rightarrow *FT' \\mid \\varepsilon$$
$$F \\rightarrow (E) \\mid i$$

Here `i` represents an identifier. This grammar gives multiplication higher precedence than addition and supports parentheses.

In [3]:
%%writefile recursive_descent.c
#include <stdio.h>
#include <string.h>

char input[200];
int pos = 0, error = 0;

void E(void);
void Edash(void);
void T(void);
void Tdash(void);
void F(void);

void match(char expected) {
    if (input[pos] == expected) pos++;
    else error = 1;
}

void E(void) {
    if (error) return;
    T();
    Edash();
}

void Edash(void) {
    if (error) return;
    if (input[pos] == '+') {
        match('+');
        T();
        Edash();
    }
}

void T(void) {
    if (error) return;
    F();
    Tdash();
}

void Tdash(void) {
    if (error) return;
    if (input[pos] == '*') {
        match('*');
        F();
        Tdash();
    }
}

void F(void) {
    if (error) return;
    if (input[pos] == 'i') {
        match('i');
    } else if (input[pos] == '(') {
        match('(');
        E();
        match(')');
    } else {
        error = 1;
    }
}

int main(void) {
    printf("Grammar:\n");
    printf("E  -> TE'\nE' -> +TE' | epsilon\n");
    printf("T  -> FT'\nT' -> *FT' | epsilon\n");
    printf("F  -> (E) | i\n");
    printf("Enter an expression using i, +, *, (, and ): ");
    scanf("%199s", input);

    E();
    if (!error && input[pos] == '\0')
        printf("Accepted: the input string is valid.\n");
    else
        printf("Rejected: syntax error at position %d near '%c'.\n",
               pos + 1, input[pos] ? input[pos] : '$');
    return 0;
}


Writing recursive_descent.c


### Compile and test with a valid input

For example, `i+i*i` is generated by the grammar.

In [4]:
!gcc recursive_descent.c -o recursive_descent
!printf 'i+i*i\n' | ./recursive_descent

Grammar:
E  -> TE'
E' -> +TE' | epsilon
T  -> FT'
T' -> *FT' | epsilon
F  -> (E) | i
Enter an expression using i, +, *, (, and ): Accepted: the input string is valid.


### Expected output for the valid input

```text
Grammar:
E  -> TE'
E' -> +TE' | epsilon
T  -> FT'
T' -> *FT' | epsilon
F  -> (E) | i
Enter an expression using i, +, *, (, and ): Accepted: the input string is valid.
```

### Test with another input

Change the value below to test a different expression. The example `i+*i` is invalid because an operand is missing after `+`.

In [5]:
expression = 'i+*i'  #@param {type:"string"}
!printf '%s\n' "$expression" | ./recursive_descent

Grammar:
E  -> TE'
E' -> +TE' | epsilon
T  -> FT'
T' -> *FT' | epsilon
F  -> (E) | i
Enter an expression using i, +, *, (, and ): Rejected: syntax error at position 3 near '*'.


## Result

The CFG was checked and immediate left factoring was applied where required. A recursive-descent parser was also implemented for the stated grammar and tested using valid and invalid input strings.